In [1]:
import sys
sys.path.append("..")  # Adds the parent directory (src) to the Python path

In [2]:
from backend.backend.utils import pod_parser

In [15]:
import requests
SERVER_URL = "http://127.0.0.1:8008"
#SERVER_URL = "http://192.168.2.239"
# gather the podcast slug, episode guid, and audio file link for transcription
podcasts = requests.get(f"{SERVER_URL}/api/podcasts/")
podcasts = podcasts.json()

In [4]:
def get_new_episodes(podcast):
    """Get the new episodes of a podcast from the RSS feed.

    Parameters
    ----------
    podcast : dict
        The podcast dictionary.

    Returns
    -------
    list
        A list of new episodes.
    """
    episodes_all = pod_parser.parse_channel(podcast["rss"])
    episodes_db = [entry["guid"] for entry in podcast["audioitem_set"]]

    # find the highest index in episodes_all that has a guid matching any guid in db_episodes
    #inefficient, but works
    idx = 0
    for i, episode in enumerate(episodes_all["audioitem_set"]):
        if episode["guid"] in episodes_db:
            idx = i

    # with the oldest guid in the db as starting point, find episodes on the RSS feed not in the db
    new_guids = set([ep["guid"] for ep in episodes_all["audioitem_set"][:idx]]) - set(episodes_db)
    new_guids = list(new_guids)

    # get the full episode dictionary for each new guid
    new_eps = []
    for guid in new_guids:
        for episode in episodes_all["audioitem_set"]:
            if episode["guid"] == guid:
                new_eps.append(episode)
    return new_eps

    

In [5]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)

278

In [6]:
def get_episodes_wo_transcription(podcasts):
    """Get the episodes already in the database without transcription for each podcast.

    Parameters
    ----------
    podcasts : list
        A list of podcasts.

    Returns
    -------
    list
        A list of episodes without transcription for each podcast.
    """
    episodes_wo_transcription = []
    for podcast in podcasts:
        for audioitem in podcast['audioitem_set']:
            # check if audioitem['transcription_set'] is empty
            if len(audioitem['transcription_set']) == 0:
                episodes_wo_transcription.append((audioitem, podcast))
    return episodes_wo_transcription


In [7]:
import os

def download_audio_file(new_ep, podcast_slug):
    current_dir = os.path.dirname(os.getcwd()) # get parent of current directory
    media_dir = current_dir + "/media"

    link = new_ep.get("audio_link")
    guid = new_ep.get("guid")
    fileroot = f"{media_dir}/{podcast_slug}_{guid}"
    # check for wav file
    if not os.path.exists(f"{fileroot}.wav"):
        # check for mp3 file
        if not os.path.exists(f"{fileroot}.mp3"):
            # download mp3 file
            print(f"Downloading {fileroot}")
            !curl -o '{fileroot + ".mp3"}' -L -J '{link}'
        # convert mp3 to wav
        !ffmpeg -i '{fileroot + ".mp3"}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{fileroot + ".wav"}'
        print(f"Downloaded {fileroot}")
    else:
        print(f"File {fileroot} already exists")


    filepath = fileroot + ".wav"
    return filepath

Fetch new episodes for existing podcasts

In [8]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy

# update podcasts that are already in the database with new episodes
for podcast in podcasts:
    slug = podcast["slug"]
    print(slug)
    
    new_eps = get_new_episodes(podcast)
    files = []
    for ep in new_eps:
        print(ep.get("title"))
        # get file
        file = download_audio_file(ep, slug)
        # run transcription
        lang = podcast["language"][0:2]
        script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

        # post episode to api
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/episodes/", json=ep)
        print(res.status_code)

        # post transcription to api
        transcription_dict = script["transcription"]
        guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
        transcription_dict["guid"] = guid
        res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
        print(res.status_code)

        # post whisper default segmentation
        segmentation_dict = script["segmentation"]
        trans_uuid = res.json().get("uuid")
        segmentation_dict["uuid"] = trans_uuid
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
        print(res.status_code)

        # use spacy to split the text into sentences
        try :
            utterances = sentence_splitter.sentence_splitter(transcription_dict, "en_core_web_lg" if lang == "en" else "nb_core_news_lg")

            segmentation_dict_spacy = {
                "uuid": trans_uuid,
                "name": "spaCy",
                "segmentor": {"name": "spaCy", "version": spacy.__version__},
                "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
            }

            res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
            print("spaCy", res.status_code)
        except:
            print("spaCy failed")



/home/adamj/anaconda3/envs/whspr/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


verdict-with-ted-cruz
leger-om-livet
norsken-svensken-og-dansken
huberman-lab
the-ben-shapiro-show
the-megyn-kelly-show
the-daily
checks-and-balance-from-the-economist
dateline-nbc
pod-save-america
freakonomics-radio
logbuchnetzpolitik
apokalypse-filterkaffee
lage-der-nation-der-politik-podcast-aus-berlin
inside-europe-deutsche-welle
forklart
oppdatert
usapodden
det-store-bildet
best-of-the-left-progressive-politics-and-culture
gaslit-nation-with-andrea-chalupa-and-sarah-kendzi
the-new-abnormal
the-fox-news-rundown
the-dan-bongino-show
aftenpodden-usa
leading
the-rest-is-politics
dagens-eko
the-doctors-farmacy-with-mark-hyman-md
ancient-health-podcast
zoe-science-nutrition
stetoskopet-tidsskriftets-podkast
just-one-thing-with-michael-mosley


In [ ]:
# RSS for 20 podcasts
RSS = {
    # Verdict with Ted Cruz
    #"https://www.omnycontent.com/d/playlist/e73c998e-6e60-432f-8610-ae210140c5b1/2bee9419-43de-46ce-8996-af2a01167517/84cf551f-a33b-41d8-b112-af2a01167541/podcast.rss": "en",
    # Ta Kommandoen med Geir Aker
    #"https://feeds.acast.com/public/shows/63f77218886da700110bf9f9": "no",
    # Misjonen med Antonsen og Golden
    #"https://smartpod.no/feed/misjonen": "no",
    # Leger om livet
    #"https://feeds.acast.com/public/shows/5f883c4673174a1b3a21b64b": "no",
    # Norsken, svensken og dansken
    #"https://podkast.nrk.no/program/norsken_svensken_og_dansken.rss": "no",
    # Burde vært pensum
    #"https://podkast.nrk.no/program/burde_vaert_pensum.rss": "no",
    # Huberman Lab
    #"https://feeds.megaphone.fm/hubermanlab": "en",
    # Stuff You Should Know
    #"https://www.omnycontent.com/d/playlist/e73c998e-6e60-432f-8610-ae210140c5b1/a91018a4-ea4f-4130-bf55-ae270180c327/44710ecc-10bb-48d1-93c7-ae270180c33e/podcast.rss": "en",
    # The Ben Shapiro Show
    #"https://feeds.megaphone.fm/WWO8086402096": "en",
    # The Megyn Kelly Show  
    #"https://feeds.simplecast.com/RV1USAfC": "en",
    # Pivot
    #"https://feeds.megaphone.fm/pivot": "en",
    # The Daily
    #"https://feeds.simplecast.com/54nAGcIl": "en",
    # The Ezra Klein Show
    #"https://feeds.simplecast.com/82FI35Px": "en",
    # Lex Fridman Podcast
    #"https://lexfridman.com/feed/podcast/": "en",
    # Checks and Balance from The Economist
    #"https://rss.acast.com/checksandbalance": "en",
    # Money Talks from The Economist
    #"https://rss.acast.com/theeconomistmoneytalks": "en",
    # The Economist Asks
    #"https://rss.acast.com/theeconomistasks": "en",
    # The Ramsey Show
    #"https://feeds.megaphone.fm/RM4031649020": "en",
    # Dateline NBC
    #"https://podcastfeeds.nbcnews.com/HL4TzgYC": "en",
    # Pod Save America
    #"https://feeds.simplecast.com/dxZsm5kX": "en",
    # Freakonomics Radio
    #"https://feeds.simplecast.com/Y8lFbOT4": "en",

}

Add a new podcast

In [ ]:
#new_eps = ["https://feeds.metaebene.me/lnp/m4a"]
#new_eps = ["https://apokalypse-und-filterkaffee.podigee.io/feed/mp3"]
#new_eps = ["https://feeds.lagedernation.org/feeds/ldn-mp3.xml", "https://rss.dw.com/xml/podcast_inside-europe", "https://podcast.stream.schibsted.media/ap/100194", "https://podkast.nrk.no/program/oppdatert.rss", "https://api.sr.se/api/rss/pod/22712", "https://api.dr.dk/podcasts/v1/feeds/genstart.xml?format=podcast", "https://rss.podplaystudio.com/692.xml"]
#new_eps = ["https://rss.art19.com/the-al-franken-podcast"]
#new_eps = ["https://feeds.megaphone.fm/PPY5667678674", "https://access.acast.com/rss/the-new-abnormal/default" ]
#new_eps = ["https://hippiesympathizer.libsyn.com/mp3.rss", "https://gaslitnation.libsyn.com/rss",  ]

In [ ]:
#new_eps = [
#    "https://feeds.megaphone.fm/FOXM1880458659", # fox news rundown
#    "https://feeds.megaphone.fm/WWO3519750118", # the dan bongino show
#]
#new_eps = [
#    "https://podcast.stream.schibsted.media/ap/100196", # Aftenpodden USA
#    "https://feeds.acast.com/public/shows/63c14d942c42340011fc0c8e", #leading
#    "https://feeds.acast.com/public/shows/620cc95e2641e200137b94d8", # rest is politics
#]


In [ ]:
#new_eps = [
#    "https://api.sr.se/api/rss/pod/itunes/41978", # dagens eko
#    "https://api.dr.dk/podcasts/v1/feeds/genstart.xml?format=podcast", # genstart
#]

In [9]:
##new_eps = [
#    "https://feeds.acast.com/public/shows/5aecaca3a15c2dd12887881a", # doctor's farmacy
##    "https://www.spreaker.com/show/4186920/episodes/feed", # ancient health
 #   "https://feeds.captivate.fm/zoe_science_and_nutrition/", # zoe science and nutrition
#    "https://feed.pod.space/stetoskopet", # stetoskopet legeforeningen
#    "https://podcasts.files.bbci.co.uk/p09by3yy.rss"
#]

new_eps = [
    "https://api.dr.dk/podcasts/v1/feeds/genstart.xml?format=podcast", # genstart
]

In [13]:
import requests
# Add each podcast in RSS to database via API
# the API will automatically download some number of recent episodes

for podcast in new_eps:
  print(podcast)
  res = requests.post(f"{SERVER_URL}/api/podcasts/", data={
    "rss": podcast
  })
  print(res.status_code)

https://api.dr.dk/podcasts/v1/feeds/genstart.xml?format=podcast
201


In [11]:
res.text

'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta http-equiv="content-type" content="text/html; charset=utf-8">\n  <meta name="robots" content="NONE,NOARCHIVE">\n  <title>AttributeError\n          at /api/podcasts/</title>\n  <style type="text/css">\n    html * { padding:0; margin:0; }\n    body * { padding:10px 20px; }\n    body * * { padding:0; }\n    body { font:small sans-serif; background-color:#fff; color:#000; }\n    body>div { border-bottom:1px solid #ddd; }\n    h1 { font-weight:normal; }\n    h2 { margin-bottom:.8em; }\n    h3 { margin:1em 0 .5em 0; }\n    h4 { margin:0 0 .5em 0; font-weight: normal; }\n    code, pre { font-size: 100%; white-space: pre-wrap; word-break: break-word; }\n    summary { cursor: pointer; }\n    table { border:1px solid #ccc; border-collapse: collapse; width:100%; background:white; }\n    tbody td, tbody th { vertical-align:top; padding:2px 3px; }\n    thead th {\n      padding:1px 6px 1px 3px; background:#fefefe; text-align:left;\n      font-weig

In [ ]:
res.status_code 

In [ ]:
res.text

In [ ]:
res.text

In [16]:
eps = get_episodes_wo_transcription(podcasts)
len(eps)

5

In [ ]:
len(eps)

In [17]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy
from json import JSONDecodeError

eps = get_episodes_wo_transcription(podcasts)

# update podcasts that are already in the database with new episodes

for episode in eps:
    podcast = episode[1]
    ep = episode[0]
    slug = podcast.get("slug")
    print(ep.get("title"))
    # get file
    file = download_audio_file(ep, slug)
    # run transcription
    lang = podcast["language"][0:2]
    script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

    # post transcription to api
    transcription_dict = script["transcription"]
    guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
    transcription_dict["guid"] = guid
    res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
    print(res.status_code)

    # post whisper default segmentation
    segmentation_dict = script["segmentation"]
    trans_uuid = res.json().get("uuid")
    segmentation_dict["uuid"] = trans_uuid
    res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
    print(res.status_code)

    # use spacy to split the text into sentences
    spacy_dict = {"en": "en_core_web_lg", "no": "nb_core_news_lg", "de": "de_dep_news_trf", "se": "sv_core_news_lg", "da": "da_core_news_trf"}
    try :
        utterances = sentence_splitter.sentence_splitter(transcription_dict, spacy_dict[lang])

        segmentation_dict_spacy = {
            "uuid": trans_uuid,
            "name": "spaCy",
            "segmentor": {"name": "spaCy", "version": spacy.__version__},
            "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
        }

        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
        print("spaCy", res.status_code)
    except:
        print("spaCy failed")

Ikke for specielle børn
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 36.1M  100 36.1M    0     0  42.0M      0 --:--:-- --:--:-- --:--:-- 58.4M
ffmpeg version 4.3 Copyright (c) 2000-2020 the FFmpeg developers
  built with gcc 7.3.0 (crosstool-NG 1.23.0.449-a04d0)
  configuration: --prefix=/opt/conda/conda-bld/ffmpeg_1597178665428/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placeh --cc=/opt/conda/conda-bld/ffmpeg_1597178665428/_build_env/bin/x86_64-conda_cos6-linux-gnu-cc --disable-doc --disable-openssl --enable-avresample --enable-gnutls --enable-hardcoded-tables --enable-libfreetype --enable-libopenh264 --enable-pic --enable-pthread

In [ ]:
res.text

In [ ]:
res.text